# Cue-words summary

Builds the per-sample cue-words table (grouped by semantic category, translated to English) and
exports it as CSV and LaTeX. Requires network access for `deep_translator`.

In [ ]:
import numpy as np
import pandas as pd
from deep_translator import GoogleTranslator

from config import RESSERENO_ROOT, MATH_ANXIETY_ROOT

FORMATTING = RESSERENO_ROOT / "out" / "formatting"
MA_FORMATTING = MATH_ANXIETY_ROOT / "out" / "formatting"

## Load the association files

In [ ]:
assoc_2023 = pd.read_csv(MA_FORMATTING / "humans_2023" / "associations_valence_2023_humans.csv")
assoc_2024 = pd.read_csv(MA_FORMATTING / "humans_2024" / "associations_valence_2024_humans.csv")
assoc_BA_2024 = pd.read_csv(FORMATTING / "psychology_BA_2024" / "associations_valence_2024.csv")

assoc_2025 = pd.read_csv(FORMATTING / "all_students_2025" / "associations_valence_task_2025.csv")
assoc_psychology_2025 = assoc_2025[assoc_2025["participant_id"].str.startswith("interfacce")]

## Semantic categories

In [ ]:
# Italian cue words grouped by semantic category (2023-2025 samples)
CATEGORY_LISTS = {
    "STEM Disciplines": [
        "algoritmo", "analisi", "biologia", "calcolo", "dimostrazione", "equazione", "fisica",
        "informatica", "ingegneria", "matematica", "matematico/a", "numeri",
        "performance_in_matematica", "statistica", "tecnologia", "test_di_matematica",
        "utilità_della_matematica", "dati", "grafico", "ipotesi", "laboratorio", "logica",
        "ricerca", "scienza", "scienziata", "scienziato", "scoperta", "esperimento",
        "innovazione", "neuroscienze",
    ],
    "Academic Environment": [
        "classe", "compagni", "docente", "genitore", "genitori", "insegnante", "studente",
        "studente/ssa", "professore", "lavagna", "mappa_concettuale", "note", "scuola",
        "università", "laurea",
    ],
    "Evaluation": ["esame", "esercitazione", "prova", "test", "media", "valutazione", "voto"],
    "Learning and Motivation": [
        "cambio_di_piani", "compito", "concentrazione", "confusione", "conoscenza", "creatività",
        "curiosità", "distrazione", "divertimento", "errori", "fissare_un_obiettivo", "impegno",
        "importanza", "intelligenza", "memorizzare", "motivazione", "pianificare", "problema",
        "programma", "programmare", "programmazione", "revisionare", "rimandare", "scadenza",
        "sfida", "sforzo", "soluzione", "studiare", "successo", "tenere_traccia", "futuro",
        "modello", "sapere", "regole", "relazione", "sviluppo", "passione", "lavoro",
    ],
    "Mental Health": [
        "ansia", "arte", "attaccamento", "benessere", "cognizione", "counselling", "emozione",
        "fallimento", "frustrazione", "imbarazzo", "meditazione", "mente", "noia", "panico",
        "paura", "personalità", "psicologia", "psicopatologia", "psicoterapia", "terapeuta",
        "terapia", "stress", "comportamento", "vita",
    ],
}

# 2019 experts sample: already-English cue words per category (no translation needed)
CATEGORY_LISTS_2019 = {
    "STEM Disciplines": [
        "science", "physics", "mathematics", "chemistry", "biology", "statistics", "electricity",
        "computer", "evolution", "dna", "experiment", "cell", "derivative", "space", "graph",
        "theorem", "trigonometry", "integral", "model", "complex", "chaos", "solid", "number",
        "hub", "system", "network", "theory",
    ],
    "Academic Environment": [
        "university", "school", "conference", "lab", "studies", "teaching", "student", "degree",
        "mathematician", "prisoner",
    ],
    "Evaluation": ["analysis", "game", "word", "loop", "market"],
    "Learning and Motivation": ["language", "person", "art"],
    "Mental Health": ["life", "brain", "water", "scientific", "cloud"],
}
CATEGORY_LISTS_2019 = {k: [w.capitalize() for w in v] for k, v in CATEGORY_LISTS_2019.items()}

## Build the per-sample category table

In [ ]:
def cue_words_summary(df):
    """One-row dataframe: for each category, the (English) cue words present in `df`."""
    present = set(df["cue_word"])
    grouped = {cat: [w for w in words if w in present] for cat, words in CATEGORY_LISTS.items()}
    translated = {
        cat: [GoogleTranslator(source="it", target="en").translate(w.capitalize()) for w in words]
        for cat, words in grouped.items()
    }
    n_words = sum(len(v) for v in grouped.values())
    print(f"{n_words} cue words for {df['participant_id'].nunique()} participants")
    return pd.DataFrame([translated])


rows = [
    pd.DataFrame([CATEGORY_LISTS_2019]),   # 2019 experts (already English)
    cue_words_summary(assoc_2023),
    cue_words_summary(assoc_2024),
    cue_words_summary(assoc_BA_2024),
    cue_words_summary(assoc_psychology_2025),
]
df_summary = pd.concat(rows, axis=0, ignore_index=True)

## Format, count and export

In [ ]:
def count_words(cell):
    if isinstance(cell, (list, np.ndarray)):
        return len(cell)
    if pd.isna(cell) or str(cell).strip() == "":
        return 0
    return len([w for w in str(cell).split(",") if w.strip()])


# join the per-category word lists into comma-separated strings
for col in df_summary.columns:
    df_summary[col] = df_summary[col].apply(lambda x: ", ".join(x) if isinstance(x, list) else x)

df_summary.insert(0, "Set", df_summary.index + 1)
df_summary["Total"] = df_summary[[c for c in df_summary.columns if c != "Set"]].apply(
    lambda row: sum(count_words(cell) for cell in row), axis=1
)

df_summary.to_csv(FORMATTING / "cue_words_summary.csv", index=False)
df_summary

In [ ]:
latex = (
    df_summary.to_latex(
        multirow=True,
        position="!hbpt",
        caption="Cue words for different samples.",
        label="tab: cue_words_summary",
        column_format="c|p{2.8cm}|p{3.2cm}|p{2.3cm}|p{3.2cm}|p{2.8cm}",
        index=False,
    )
    .replace("_", " ")
    .replace("NaN", "")
)
latex = latex.replace(r"\begin{table}[!hbpt]", "\\begin{table}[!hbpt]\n\\scriptsize\n\\centering")

with open(FORMATTING / "cue_words_summary_latex.txt", "w") as f:
    f.write(latex)